In [1]:
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.regularizers import l1_l2

2024-11-08 18:00:18.678234: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-08 18:00:18.709112: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-11-08 18:00:19.193904: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
def conv3x3(x, out_planes, stride=1, name=None):
    x = ZeroPadding2D(padding=1, name=f'{name}_pad')(x)
    return Conv2D(filters=out_planes, kernel_size=3, strides=stride, use_bias=False, kernel_initializer="glorot_uniform", name=name)(x)

def basic_block(x, planes, stride=1, downsample=None, name=None):
    identity = x

    out = conv3x3(x, planes, stride=stride, name=f'{name}.conv1')
    out = BatchNormalization(momentum=0.9, epsilon=1e-5, name=f'{name}.bn1')(out)
    out = ReLU(name=f'{name}.relu1')(out)

    out = conv3x3(out, planes, name=f'{name}.conv2')
    out = BatchNormalization(momentum=0.9, epsilon=1e-5, name=f'{name}.bn2')(out)

    if downsample is not None:
        for layer in downsample:
            identity = layer(identity)

    out = Add(name=f'{name}.add')([identity, out])
    out = ReLU(name=f'{name}.relu2')(out)

    return out

def make_layer(x, planes, blocks, stride=1, name=None):
    downsample = None
    inplanes = x.shape[3]
    if stride != 1 or inplanes != planes:
        downsample = [
            Conv2D(filters=planes, kernel_size=1, strides=stride, use_bias=False, kernel_initializer='glorot_uniform', name=f'{name}.0.downsample.0'),
            BatchNormalization(momentum=0.9, epsilon=1e-5, name=f'{name}.0.downsample.1'),
        ]

    x = basic_block(x, planes, stride, downsample, name=f'{name}.0')
    for i in range(1, blocks):
        x = basic_block(x, planes, name=f'{name}.{i}')

    return x

def resnet(blocks_per_layer, num_classes=1000):
    input_layer = Input((224,224,3))
    x = ZeroPadding2D(padding=3, name='conv1_pad')(input_layer)
    x = Conv2D(filters=64, kernel_size=7, strides=2, use_bias=False, kernel_initializer='glorot_uniform', name='conv1')(x)
    x = BatchNormalization(momentum=0.9, epsilon=1e-5, name='bn1')(x)
    x = ReLU(name='relu1')(x)
    x = ZeroPadding2D(padding=1, name='maxpool_pad')(x)
    x = MaxPool2D(pool_size=3, strides=2, name='maxpool')(x)

    x = make_layer(x, 64, blocks_per_layer[0], name='layer1')
    x = make_layer(x, 128, blocks_per_layer[1], stride=2, name='layer2')
    x = make_layer(x, 256, blocks_per_layer[2], stride=2, name='layer3')
    x = make_layer(x, 512, blocks_per_layer[3], stride=2, name='layer4')

    x = GlobalAveragePooling2D(name='avgpool')(x)
    regularization_factor = 0.01
    x = Dense(units=256, activation = 'relu', name='Dense', kernel_regularizer = l1_l2(regularization_factor))(x)
    x = Dense(units=num_classes, activation = 'softmax', name='fc')(x)
    model = tf.keras.models.Model(inputs = input_layer, outputs = x)
    return model

In [ ]:
resnet18 = resnet([2,2,2,2],7)
resnet18.summary()

In [ ]:
resnet18.save('./public/resnet18.h5')

In [ ]:
efficientnet = tf.keras.applications.EfficientNetB0(weights = 'imagenet',
                                                    include_top = False)

efficientnet.summary()

In [ ]:
from tensorflow.keras import layers, models, Sequential
import tensorflow as tf

# EfficientNet 모델 불러오기
efficientnet = tf.keras.applications.EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

new_model = Sequential([])
for i, layer in enumerate(efficientnet.layers):
    if i == 1 or i == 2 or i==3:
        print(layer.name)
    

# new_model.summary()  # 모델 요약 출력 (Normalization 레이어가 삭제된 상태)

In [ ]:
efficientnet.save('./public/efficientnet.h5')

In [ ]:
nasnet = tf.keras.applications.NASNetMobile()
nasnet.summary()

In [ ]:
nasnet.save('./public/nasnet.h5')

In [ ]:
inputs= tf.keras.layers.Input((224,224,3))
x = tf.keras.layers.Conv2D(filters=64, kernel_size=10, padding='valid')(inputs)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPool2D((2,2))(x)

x = tf.keras.layers.Conv2D(filters=128, kernel_size=7, padding='valid')(inputs)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPool2D((2,2))(x)

x = tf.keras.layers.Conv2D(filters=128, kernel_size=4, padding='valid')(inputs)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPool2D((2,2))(x)

x = tf.keras.layers.Conv2D(filters=128, kernel_size=4, padding='valid')(inputs)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPool2D((2,2))(x)

x = tf.keras.layers.Flatten()(x)

x = tf.keras.layers.Dense(256, activation = 'relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)

output = tf.keras.layers.Dense(1, activation = 'sigmoid')(x)

siamesenet = tf.keras.models.Model(inputs = inputs, outputs = output)

siamesenet.summary()